In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import math
import random
import seaborn as sns
import matplotlib.pyplot as plt
# Sirve para medir la velocidad de una función
import time
%load_ext line_profiler
import sys
sys.path.insert(1, '/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Codigos/Funciones_utiles')
import funciones_aux_bootstrap as fab
from matplotlib import legend_handler

# Fijamos el estilo de la gráfica
plt.style.use('petroff10')

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


# Casillas luego Votos método 2 vs Casillas

## No estratificado

In [3]:
# Leemos las bases

# Base de bootstrap por casillas sin estratificar
df_boostraps_c_s_e=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases finales rep/Bootstrap por casillas/Distintas_muestras_casillas_bootstrap_sin_est_f_1.csv", index_col=0)
# Base de bootstrap por votos (directo)
df_boostraps_votos_se=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases finales rep/Bootstrap votos/Metodo 1/Distintas_muestras_submuestras_bootstrap_votos_met_1_sin_est_f_1.csv", index_col=0)

## Tabla candidatos con menor cobertura

In [92]:
# Para casillas
df_boostraps_c_m_c=df_boostraps_c_s_e[["Cobertura_m_candidato_nom", "Tamaño_muestra_casillas"]].copy()
df_boostraps_c_m_c["Tamaño_muestra_casillas"]=df_boostraps_c_m_c["Tamaño_muestra_casillas"].astype(int)
df_agg_cand_c=df_boostraps_c_m_c.groupby('Cobertura_m_candidato_nom').count().reset_index()
# Creamos una columna para identificar que se está muestreando
df_agg_cand_c["Sampling method"]='Polling places'
# Porcentaje del total
df_agg_cand_c["Proportion of sampling method"] = df_agg_cand_c["Tamaño_muestra_casillas"] / df_boostraps_c_m_c.shape[0]
# Quitamos la columna de Tamaño_muestra_casillas
df_agg_cand_c=df_agg_cand_c.drop(columns=["Tamaño_muestra_casillas"])
# Renombramos la columna del candidato
df_agg_cand_c=df_agg_cand_c.rename(columns={"Cobertura_m_candidato_nom": "Candidate with the lowest coverage"})
df_agg_cand_c

,Candidate with the lowest coverage,Sampling method,Proportion of sampling method
0,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places,1.0


In [93]:
# Para casillas luego votos
df_boostraps_v_m_c=df_boostraps_votos_se[["Cobertura_m_candidato_nom", "Tamaño_muestra_casillas"]].copy()
df_boostraps_v_m_c["Tamaño_muestra_casillas"] = df_boostraps_v_m_c["Tamaño_muestra_casillas"].astype(int) 
df_agg_cand_v=df_boostraps_v_m_c.groupby('Cobertura_m_candidato_nom').count().reset_index()
# Agregamps la columna de que se está muestreando
df_agg_cand_v["Sampling method"] = "Polling places then votes"
# Porcentaje del total
df_agg_cand_v["Proportion of sampling method"] = df_agg_cand_v["Tamaño_muestra_casillas"] / df_boostraps_v_m_c.shape[0]
# Renombramos la columna del candidato
df_agg_cand_v=df_agg_cand_v.rename(columns={"Cobertura_m_candidato_nom": "Candidate with the lowest coverage"})
# Quitamos la columna de Tamaño_muestra_casillas
df_agg_cand_v=df_agg_cand_v.drop(columns=["Tamaño_muestra_casillas"])
df_agg_cand_v

,Candidate with the lowest coverage,Sampling method,Proportion of sampling method
0,JOAQUIN_DIAZ_MENA,Polling places then votes,0.071429
1,RENAN_BARRERA_CONCHA,Polling places then votes,0.761905
2,VIDA_ARAVARI_GOMEZ_HERRERA,Polling places then votes,0.023810
3,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places then votes,0.142857


In [94]:
# Concatenamos las tablas
df_agg_com=pd.concat([df_agg_cand_c,df_agg_cand_v])
# Reordenamos las columnas
df_agg_com_f=df_agg_com[["Candidate with the lowest coverage","Sampling method","Proportion of sampling method"]].reset_index(drop=True)
df_agg_com_f

,Candidate with the lowest coverage,Sampling method,Proportion of sampling method
0,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places,1.000000
1,JOAQUIN_DIAZ_MENA,Polling places then votes,0.071429
2,RENAN_BARRERA_CONCHA,Polling places then votes,0.761905
3,VIDA_ARAVARI_GOMEZ_HERRERA,Polling places then votes,0.023810
4,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places then votes,0.142857


In [96]:
# Renombramos los nombres de los candidatos
df_agg_com_f["Candidate with the lowest coverage"]=["Jasmín López Manrique", "Joaquín Díaz Mena", "Renan Barrera Concha", "Vida Gómez Herrera", "Jasmín López Manrique"]
df_agg_com_f

,Candidate with the lowest coverage,Sampling method,Proportion of sampling method
0,Jasmín López Manrique,Polling places,1.000000
1,Joaquín Díaz Mena,Polling places then votes,0.071429
2,Renan Barrera Concha,Polling places then votes,0.761905
3,Vida Gómez Herrera,Polling places then votes,0.023810
4,Jasmín López Manrique,Polling places then votes,0.142857


In [99]:
print(df_agg_com_f.to_latex(index=False, label='tab:Lowest_cov_casillas_sub_votos_vs_cas',caption='Candidates with the lowest coverage (non-stratified)'))

\begin{table}
\caption{Candidates with the lowest coverage (non-stratified)}
\label{tab:Lowest_cov_casillas_sub_votos_vs_cas}
\begin{tabular}{llr}
\toprule
Candidate with the lowest coverage & Sampling method & Proportion of sampling method \\
\midrule
Jasmín López Manrique & Polling places & 1.000000 \\
Joaquín Díaz Mena & Polling places then votes & 0.071429 \\
Renan Barrera Concha & Polling places then votes & 0.761905 \\
Vida Gómez Herrera & Polling places then votes & 0.023810 \\
Jasmín López Manrique & Polling places then votes & 0.142857 \\
\bottomrule
\end{tabular}
\end{table}



## Diferencia en cobertura máxima y mínima por candidato

In [120]:
# Base para la casillas inicial
df_aux_c=df_boostraps_c_s_e[["Cob_JOAQUIN_DIAZ_MENA","Cob_RENAN_BARRERA_CONCHA","Cob_VIDA_ARAVARI_GOMEZ_HERRERA","Cob_VOTOS_NULOS_CAND_NO_REGIS","Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE"]]
# Máxima diferencia por columna
max_diff = df_aux_c.max() - df_aux_c.min()

# Lo pasamos a Data Frame
df_max_c=pd.DataFrame(max_diff).reset_index().rename(columns={'index':'Candidates',0:'Maximum difference in coverages (polling places)'})
df_max_c

,Candidates,Maximum difference in coverages (polling places)
0,Cob_JOAQUIN_DIAZ_MENA,0.028
1,Cob_RENAN_BARRERA_CONCHA,0.031
2,Cob_VIDA_ARAVARI_GOMEZ_HERRERA,0.050
3,Cob_VOTOS_NULOS_CAND_NO_REGIS,0.028
4,Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE,0.164


In [121]:
# Base para la votos inicial
df_aux_v=df_boostraps_votos_se[["Cob_JOAQUIN_DIAZ_MENA","Cob_RENAN_BARRERA_CONCHA","Cob_VIDA_ARAVARI_GOMEZ_HERRERA","Cob_VOTOS_NULOS_CAND_NO_REGIS","Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE"]]
# Máxima diferencia por columna
max_diff_v = df_aux_v.max() - df_aux_v.min()

# Lo pasamos a Data Frame
df_max_v=pd.DataFrame(max_diff_v).reset_index().rename(columns={'index':'Candidates',0:'Maximum difference in coverages (votes)'})
df_max_v

,Candidates,Maximum difference in coverages (votes)
0,Cob_JOAQUIN_DIAZ_MENA,0.014
1,Cob_RENAN_BARRERA_CONCHA,0.021
2,Cob_VIDA_ARAVARI_GOMEZ_HERRERA,0.018
3,Cob_VOTOS_NULOS_CAND_NO_REGIS,0.018
4,Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE,0.051


In [124]:
# Juntamos las bases
df_max_f=df_max_c.merge(df_max_v, how='inner', on='Candidates')
# Renombramos la columna de candidatos
df_max_f["Candidates"]=["Joaquín Díaz Mena", "Renan Barrera Concha", "Vida Gómez Herrera", "Null votes and votes for unregistered candidates","Jasmín López Manrique"]
df_max_f

,Candidates,Maximum difference in coverages (polling places),Maximum difference in coverages (votes)
0,Joaquín Díaz Mena,0.028,0.014
1,Renan Barrera Concha,0.031,0.021
2,Vida Gómez Herrera,0.050,0.018
3,Null votes and votes for unregistered candidates,0.028,0.018
4,Jasmín López Manrique,0.164,0.051


In [125]:
print(df_max_f.to_latex(index=False, label='tab:Max_diff_cov_votos_vs_cas',caption='Maximum difference in coverages'))

\begin{table}
\caption{Maximum difference in coverages}
\label{tab:Max_diff_cov_votos_vs_cas}
\begin{tabular}{lrr}
\toprule
Candidates & Maximum difference in coverages (polling places) & Maximum difference in coverages (votes) \\
\midrule
Joaquín Díaz Mena & 0.028000 & 0.014000 \\
Renan Barrera Concha & 0.031000 & 0.021000 \\
Vida Gómez Herrera & 0.050000 & 0.018000 \\
Null votes and votes for unregistered candidates & 0.028000 & 0.018000 \\
Jasmín López Manrique & 0.164000 & 0.051000 \\
\bottomrule
\end{tabular}
\end{table}



## Estratificado por distritos locales

In [3]:
# Leemos las bases

# Base de bootstrap por casillas sin estratificar
df_boostraps_c_est=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases finales rep/Bootstrap por casillas/Distintas_muestras_casillas_bootstrap_estratificado_f_1.csv", index_col=0)
# Base de bootstrap por votos (submuestreo) método 1
df_boostraps_votos_est=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases finales rep/Bootstrap votos/Metodo 1/Distintas_muestras_submuestras_bootstrap_votos_met_1_estratificado_f_1.csv", index_col=0)


## Tabla candidatos con menor cobertura

In [6]:
# Para casillas
df_boostraps_c_m_est=df_boostraps_c_est[["Cobertura_m_candidato_nom", "Tamaño_muestra_casillas"]].copy()
df_boostraps_c_m_est["Tamaño_muestra_casillas"]=df_boostraps_c_m_est["Tamaño_muestra_casillas"].astype(int)
df_agg_cand_c_est=df_boostraps_c_m_est.groupby('Cobertura_m_candidato_nom').count().reset_index()
# Creamos una columna para identificar que se está muestreando
df_agg_cand_c_est["Sampling method"]='Polling places'
# Porcentaje del total
df_agg_cand_c_est["Proportion of sampling method"] = df_agg_cand_c_est["Tamaño_muestra_casillas"] / df_boostraps_c_m_est.shape[0]
# Quitamos la columna de Tamaño_muestra_casillas
df_agg_cand_c_est=df_agg_cand_c_est.drop(columns=["Tamaño_muestra_casillas"])
# Renombramos la columna del candidato
df_agg_cand_c_est=df_agg_cand_c_est.rename(columns={"Cobertura_m_candidato_nom": "Candidate with the lowest coverage"})
df_agg_cand_c_est

,Candidate with the lowest coverage,Sampling method,Proportion of sampling method
0,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places,1.0


In [8]:
# Para casillas luego votos
df_boostraps_v_m_c_est=df_boostraps_votos_est[["Cobertura_m_candidato_nom", "Tamaño_muestra_casillas"]].copy()
df_boostraps_v_m_c_est["Tamaño_muestra_casillas"] = df_boostraps_v_m_c_est["Tamaño_muestra_casillas"].astype(int) 
df_agg_cand_v_est=df_boostraps_v_m_c_est.groupby('Cobertura_m_candidato_nom').count().reset_index()
# Agregamps la columna de que se está muestreando
df_agg_cand_v_est["Sampling method"] = "Polling places then votes"
# Porcentaje del total
df_agg_cand_v_est["Proportion of sampling method"] = df_agg_cand_v_est["Tamaño_muestra_casillas"] / df_boostraps_v_m_c_est.shape[0]
# Renombramos la columna del candidato
df_agg_cand_v_est=df_agg_cand_v_est.rename(columns={"Cobertura_m_candidato_nom": "Candidate with the lowest coverage"})
# Quitamos la columna de Tamaño_muestra_casillas
df_agg_cand_v_est=df_agg_cand_v_est.drop(columns=["Tamaño_muestra_casillas"])
df_agg_cand_v_est

,Candidate with the lowest coverage,Sampling method,Proportion of sampling method
0,JOAQUIN_DIAZ_MENA,Polling places then votes,0.071429
1,RENAN_BARRERA_CONCHA,Polling places then votes,0.595238
2,VIDA_ARAVARI_GOMEZ_HERRERA,Polling places then votes,0.095238
3,VOTOS_NULOS_CAND_NO_REGIS,Polling places then votes,0.047619
4,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places then votes,0.190476


In [14]:
# Concatenamos las tablas
df_agg_com_est=pd.concat([df_agg_cand_c_est,df_agg_cand_v_est])
# Reordenamos las columnas
df_agg_com_f_est=df_agg_com_est[["Candidate with the lowest coverage","Sampling method","Proportion of sampling method"]].reset_index(drop=True)
# Redondeamos a 4 dígitos
df_agg_com_f_est["Proportion of sampling method"] = df_agg_com_f_est["Proportion of sampling method"].round(4)
df_agg_com_f_est

,Candidate with the lowest coverage,Sampling method,Proportion of sampling method
0,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places,1.0000
1,JOAQUIN_DIAZ_MENA,Polling places then votes,0.0714
2,RENAN_BARRERA_CONCHA,Polling places then votes,0.5952
3,VIDA_ARAVARI_GOMEZ_HERRERA,Polling places then votes,0.0952
4,VOTOS_NULOS_CAND_NO_REGIS,Polling places then votes,0.0476
5,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places then votes,0.1905


In [16]:
# Renombramos los nombres de los candidatos
df_agg_com_f_est["Candidate with the lowest coverage"]=["Jasmín López Manrique","Joaquín Díaz Mena", 'Renan Barrera Concha', "Vida Gómez Herrera", "Null votes and votes for unregistered candidates", "Jasmín López Manrique"]
df_agg_com_f_est

,Candidate with the lowest coverage,Sampling method,Proportion of sampling method
0,Jasmín López Manrique,Polling places,1.0000
1,Joaquín Díaz Mena,Polling places then votes,0.0714
2,Renan Barrera Concha,Polling places then votes,0.5952
3,Vida Gómez Herrera,Polling places then votes,0.0952
4,Null votes and votes for unregistered candidates,Polling places then votes,0.0476
5,Jasmín López Manrique,Polling places then votes,0.1905


In [17]:
print(df_agg_com_f_est.to_latex(index=False, label='tab:Max_diff_cov_votos_vs_cas',caption='Maximum difference in coverages'))

\begin{table}
\caption{Maximum difference in coverages}
\label{tab:Max_diff_cov_votos_vs_cas}
\begin{tabular}{llr}
\toprule
Candidate with the lowest coverage & Sampling method & Proportion of sampling method \\
\midrule
Jasmín López Manrique & Polling places & 1.000000 \\
Joaquín Díaz Mena & Polling places then votes & 0.071400 \\
Renan Barrera Concha & Polling places then votes & 0.595200 \\
Vida Gómez Herrera & Polling places then votes & 0.095200 \\
Null votes and votes for unregistered candidates & Polling places then votes & 0.047600 \\
Jasmín López Manrique & Polling places then votes & 0.190500 \\
\bottomrule
\end{tabular}
\end{table}



## Diferencia en cobertura máxima y mínima por candidato

In [13]:
# Base para la casillas inicial
df_aux_c_est=df_boostraps_c_est[["Cob_JOAQUIN_DIAZ_MENA","Cob_RENAN_BARRERA_CONCHA","Cob_VIDA_ARAVARI_GOMEZ_HERRERA","Cob_VOTOS_NULOS_CAND_NO_REGIS","Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE"]]
# Máxima diferencia por columna
max_diff_est = df_aux_c_est.max() - df_aux_c_est.min()

# Lo pasamos a Data Frame
df_max_c_est=pd.DataFrame(max_diff_est).reset_index().rename(columns={'index':'Candidates',0:'Maximum difference in coverages (polling places)'})
df_max_c_est

,Candidates,Maximum difference in coverages (polling places)
0,Cob_JOAQUIN_DIAZ_MENA,0.081
1,Cob_RENAN_BARRERA_CONCHA,0.077
2,Cob_VIDA_ARAVARI_GOMEZ_HERRERA,0.109
3,Cob_VOTOS_NULOS_CAND_NO_REGIS,0.094
4,Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE,0.200


In [14]:
# Base para la votos inicial
df_aux_v_est=df_boostraps_votos_est[["Cob_JOAQUIN_DIAZ_MENA","Cob_RENAN_BARRERA_CONCHA","Cob_VIDA_ARAVARI_GOMEZ_HERRERA","Cob_VOTOS_NULOS_CAND_NO_REGIS","Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE"]]
# Máxima diferencia por columna
max_diff_v_est = df_aux_v_est.max() - df_aux_v_est.min()

# Lo pasamos a Data Frame
df_max_v_est=pd.DataFrame(max_diff_v_est).reset_index().rename(columns={'index':'Candidates',0:'Maximum difference in coverages (votes)'})
df_max_v_est

,Candidates,Maximum difference in coverages (votes)
0,Cob_JOAQUIN_DIAZ_MENA,0.027
1,Cob_RENAN_BARRERA_CONCHA,0.026
2,Cob_VIDA_ARAVARI_GOMEZ_HERRERA,0.012
3,Cob_VOTOS_NULOS_CAND_NO_REGIS,0.045
4,Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE,0.046


In [15]:
# Juntamos las bases
df_max_f_est=df_max_c_est.merge(df_max_v_est, how='inner', on='Candidates')
# Renombramos la columna de candidatos
df_max_f_est["Candidates"]=["Joaquín Díaz Mena", "Renan Barrera Concha", "Vida Gómez Herrera", "Null votes and votes for unregistered candidates","Jasmín López Manrique"]
df_max_f_est

,Candidates,Maximum difference in coverages (polling places),Maximum difference in coverages (votes)
0,Joaquín Díaz Mena,0.081,0.027
1,Renan Barrera Concha,0.077,0.026
2,Vida Gómez Herrera,0.109,0.012
3,Null votes and votes for unregistered candidates,0.094,0.045
4,Jasmín López Manrique,0.200,0.046


In [16]:
print(df_max_f_est.to_latex(index=False, label='tab:Max_diff_cov_votos_vs_cas',caption='Maximum difference in coverages'))

\begin{table}
\caption{Maximum difference in coverages}
\label{tab:Max_diff_cov_votos_vs_cas}
\begin{tabular}{lrr}
\toprule
Candidates & Maximum difference in coverages (polling places) & Maximum difference in coverages (votes) \\
\midrule
Joaquín Díaz Mena & 0.081000 & 0.027000 \\
Renan Barrera Concha & 0.077000 & 0.026000 \\
Vida Gómez Herrera & 0.109000 & 0.012000 \\
Null votes and votes for unregistered candidates & 0.094000 & 0.045000 \\
Jasmín López Manrique & 0.200000 & 0.046000 \\
\bottomrule
\end{tabular}
\end{table}

